## Lets import the data

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
folder = '/Users/mikkelrathtornerup/Desktop/Dynamic-Programming-Project-main/Data'
filename = 'Lohano_King_Data_Cleaned.csv'

filepath = os.path.join(folder, filename)
df = pd.read_csv(filepath, sep=';')
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns.tolist())

Columns:
['Year', 'Gross Return (nominal)', 'Costs (nominal)', 'Gross Return (nominal).1', 'Costs (nominal).1', 'Unnamed: 5', 'Gross Return (nominal).2', 'Costs (nominal).2', 'Gross Return (real) in 2000 dollars', 'Costs (real) in 2000 dollars', 'P: Raup (nominal)', 'P: Taff (nominal)', 'Ratio', 'P  (nominal)', 'P  (real) in 2000 dollars', 'Implicit Price Deflator of GNP', 'RM (percent) (Real)', '(1+rate of return): (Real)']


In [3]:
df = df.rename(columns={
    'Gross Return (real) in 2000 dollars': 'Rt',
    'Costs (real) in 2000 dollars': 'Ct',
    'P  (real) in 2000 dollars': 'Pt',
    '(1+rate of return): (Real)': 'Mt',
    'P: Raup (nominal)': 'P_raup',
    'P: Taff (nominal)': 'P_taff'
})

In [4]:
# Convert to numeric
for col in ['Year', 'Rt', 'Ct', 'Pt', 'Mt', 'P_raup', 'P_taff']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [8]:
df_main = df.dropna(subset=['Year', 'Rt', 'Ct', 'Pt', 'Mt']).copy()
df_main = df_main[(df_main['Rt'] > 0) & (df_main['Pt'] > 0) & (df_main['Mt'] > 0)].copy()
df_main['ln_Rt'] = np.log(df_main['Rt'])
df_main['ln_Pt'] = np.log(df_main['Pt'])
df_main['ln_Mt'] = np.log(df_main['Mt'])
df_main['ln_Rt_lag'] = df_main['ln_Rt'].shift(1)

In [5]:
# Main sample
df = df.dropna(subset=['Year', 'Rt', 'Ct', 'Pt']).copy()
df = df[(df['Rt'] > 0) & (df['Pt'] > 0)].copy()
df = df.sort_values('Year').reset_index(drop=True)

# Logs
df['ln_Rt'] = np.log(df['Rt'])
df['ln_Pt'] = np.log(df['Pt'])

# Leads to match gstate.m
df['ln_Rt_next'] = df['ln_Rt'].shift(-1)
df['ln_Pt_next'] = df['ln_Pt'].shift(-1)

In [9]:
# 1. Return equation
# ln(R_{t+1}) = beta0 + beta1 ln(R_t) + e1_{t+1}
df_R = df[['ln_Rt_next', 'ln_Rt']].dropna().copy()

X_R = sm.add_constant(df_R['ln_Rt'])
y_R = df_R['ln_Rt_next']

model_R = sm.OLS(y_R, X_R).fit()
eps1 = model_R.resid

# 2. Price equation
# ln(P_{t+1}) = alpha0 + alpha1 ln(P_t) + alpha2 ln(R_t) + e2_{t+1}
df_P = df[['ln_Pt_next', 'ln_Pt', 'ln_Rt']].dropna().copy()

X_P = sm.add_constant(df_P[['ln_Pt', 'ln_Rt']])
y_P = df_P['ln_Pt_next']

model_P = sm.OLS(y_P, X_P).fit()
eps2 = model_P.resid

# 3. Align residuals by time index
eps_df = pd.DataFrame({
    'eps1': eps1,
    'eps2': eps2
}).dropna()

residual_cov = eps_df['eps1'].cov(eps_df['eps2'])
residual_corr = eps_df['eps1'].corr(eps_df['eps2'])

# 4. Other statistics
avg_cost_data = df['Ct'].mean()
c_total = avg_cost_data + 30

df_ratio = df.dropna(subset=['P_raup', 'P_taff']).copy()
df_ratio = df_ratio[(df_ratio['P_raup'] > 0) & (df_ratio['P_taff'] > 0)].copy()
df_ratio['splice_ratio'] = df_ratio['P_taff'] / df_ratio['P_raup']
price_splice = df_ratio['splice_ratio'].mean()

In [13]:
# Results
print('--- Estimated coefficients ---')
print(model_R.params)
print(model_P.params)

print('\n--- Summary statistics ---')
print(f'Average production cost from data: {avg_cost_data:.6f}')
print(f'Total cost parameter c (+30):      {c_total:.6f}')
print(f'Residual correlation:              {residual_corr:.6f}')
print(f'Residual covariance:               {residual_cov:.6f}')
print(f'Price splicing ratio:              {price_splice:.6f}')
print(f'Return lag coefficient:            {model_R.params["ln_Rt"]:.6f}')

--- Estimated coefficients ---
const    1.866830
ln_Rt    0.673155
dtype: float64
const    2.096796
ln_Pt    0.890573
ln_Rt   -0.229065
dtype: float64

--- Summary statistics ---
Average production cost from data: 201.094000
Total cost parameter c (+30):      231.094000
Residual correlation:              0.250003
Residual covariance:               0.004190
Price splicing ratio:              0.867253
Return lag coefficient:            0.673155


#### Targets (forund in the paper)

$$
\begin{array}{lc}
\hline
\textbf{Statistic} & \textbf{Target Value} \\
\hline
\text{Avg. Production Cost }(c) & 231\ (\text{Real2000})  \\
\text{Residual Correlation} & 0.054 \\
\text{Residual Covariance} & 0.000886 \\
\text{Price Splicing Ratio} & 0.868  \\
\ln R_t \text{ Lag Coefficient} & 0.742391 \\
\hline
\end{array}
$$

# TEST

In [ ]:
# Python
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Load data
folder = '/Users/mikkelrathtornerup/Desktop/Dynamic-Programming-Project-main/Data'
filename = 'Lohano_King_Data_Cleaned.csv'
filepath = os.path.join(folder, filename)
df = pd.read_csv(filepath, sep=';', decimal=',')
df.columns = df.columns.str.strip()

# 2. Rename columns
df = df.rename(columns={
    'Gross Return (real) in 2000 dollars': 'Rt',
    'Costs (real) in 2000 dollars': 'Ct',
    'P  (real) in 2000 dollars': 'Pt',
    '(1+rate of return): (Real)': 'Mt',
    'P: Raup (nominal)': 'P_raup',
    'P: Taff (nominal)': 'P_taff'
})

# 3. Convert to numeric
for col in ['Year', 'Rt', 'Ct', 'Pt', 'Mt', 'P_raup', 'P_taff']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Filter main sample
df = df.dropna(subset=['Year', 'Rt', 'Ct', 'Pt', 'P_raup', 'P_taff']).copy()
df = df[(df['Rt'] > 0) & (df['Pt'] > 0) & (df['P_raup'] > 0) & (df['P_taff'] > 0)].copy()
df = df.sort_values('Year').reset_index(drop=True)

# 5. Construct spliced price series
df['P_spliced'] = np.where(
    df['Year'] <= 1992,
    0.868 * df['P_raup'],
    df['P_taff']
)

# 6. Compute logs using spliced price
df['ln_Rt'] = np.log(df['Rt'])
df['ln_Pt'] = np.log(df['P_spliced'])

# 7. Next-period variables
df['ln_Rt_next'] = df['ln_Rt'].shift(-1)
df['ln_Pt_next'] = df['ln_Pt'].shift(-1)

# 8. Estimate return equation
df_R = df[['ln_Rt_next', 'ln_Rt']].dropna().copy()
X_R = sm.add_constant(df_R['ln_Rt'])
y_R = df_R['ln_Rt_next']
model_R = sm.OLS(y_R, X_R).fit()
eps1 = model_R.resid

# 9. Estimate price equation
df_P = df[['ln_Pt_next', 'ln_Pt', 'ln_Rt']].dropna().copy()
X_P = sm.add_constant(df_P[['ln_Pt', 'ln_Rt']])
y_P = df_P['ln_Pt_next']
model_P = sm.OLS(y_P, X_P).fit()
eps2 = model_P.resid

# 10. Align residuals
eps_df = pd.DataFrame({'eps1': eps1, 'eps2': eps2}).dropna()
residual_cov = eps_df['eps1'].cov(eps_df['eps2'])
residual_corr = eps_df['eps1'].corr(eps_df['eps2'])

# 11. Print results
print('--- Estimated coefficients ---')
print(model_R.params)
print(model_P.params)
print('\n--- Summary statistics ---')
print(f'Return lag coefficient:            {model_R.params["ln_Rt"]:.6f}')
print(f'Residual covariance:               {residual_cov:.6f}')
print(f'Residual correlation:              {residual_corr:.6f}')

--- Estimated coefficients ---
const   -0.343622
ln_Rt    1.039709
dtype: float64
const    0.084845
ln_Pt    0.759928
ln_Rt    0.288077
dtype: float64

--- Summary statistics ---
Return lag coefficient:            1.039709
Residual covariance:               0.000000
Residual correlation:              nan


/Users/mikkelrathtornerup/mik2024env/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
